# 01 — Ingest rostr.cc API → UC Volume

Single, parameterized notebook that lands the rostr.cc surface area for a
**seed list of artists** into a Volume landing zone. Same code runs as a
first-time backfill or a daily incremental.

## Coverage

| Group           | Routes covered                                  | Volume target |
|-----------------|-------------------------------------------------|---------------|
| Artist detail   | `GET /v1/artist/{handle}`                       | `artists/{handle}.json` |
| Per-role team   | `GET /v1/artist/{handle}/team/{ROLE}` for ROLE in `MANAGEMENT, AGENCY, RECORD_LABEL, PUBLISHER` | `team/{handle}/{role}.json` |

## Inputs

* `ARTIST_SEED_CSV` — CSV (relative or absolute path) with columns `artist_name,rostr_handle`. The handle column is optional; we slugify the name when blank.
* `ROSTR_ROLES`     — comma-separated list of company roles to pull per artist. Default: all four.

## Idempotency

Each role payload is a single JSON file; we skip re-uploading if the file
already exists in the Volume. Re-runs only call rostr.cc for missing files
— so a partially-failed run continues cleanly on the next pass.

If you need to *force* a fresh pull (e.g. weekly refresh of all data), set
`FORCE_REFRESH=true` in `.env`.

In [1]:
import os, json, csv
from concurrent.futures import ThreadPoolExecutor, as_completed
from dotenv import load_dotenv
load_dotenv()

try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

In [2]:
UC_CATALOG    = os.getenv('UC_CATALOG', 'alexander_booth')
UC_SCHEMA     = os.getenv('UC_SCHEMA',  'rostr_music_industry')
BRONZE_SCHEMA = f'{UC_SCHEMA}_bronze'
VOLUME_NAME   = 'raw_data'
VOLUME_PATH   = f'/Volumes/{UC_CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}'
ARTISTS_DIR   = f'{VOLUME_PATH}/artists'
TEAM_DIR      = f'{VOLUME_PATH}/team'

ARTIST_SEED_CSV  = os.getenv('ARTIST_SEED_CSV',  'artists_seed.csv')
ROSTR_ROLES      = [r.strip() for r in os.getenv('ROSTR_ROLES', 'MANAGEMENT,AGENCY,RECORD_LABEL,PUBLISHER').split(',') if r.strip()]
INGEST_WORKERS   = int(os.getenv('INGEST_WORKERS', '4'))
FORCE_REFRESH    = os.getenv('FORCE_REFRESH', 'false').lower() == 'true'
INGEST_RATE_LIMIT = float(os.getenv('INGEST_RATE_LIMIT', '0.5'))

print(f'Volume root  : {VOLUME_PATH}')
print(f'Roles        : {ROSTR_ROLES}')
print(f'Workers      : {INGEST_WORKERS}  (per-worker rate limit {INGEST_RATE_LIMIT}s)')
print(f'Force refresh: {FORCE_REFRESH}')

Volume root  : /Volumes/alexander_booth/rostr_music_industry_bronze/raw_data
Roles        : ['MANAGEMENT', 'AGENCY', 'RECORD_LABEL', 'PUBLISHER']
Workers      : 4  (per-worker rate limit 0.5s)
Force refresh: False


In [3]:
def get_secret(env_var: str, scope: str = 'rostr', key: str | None = None) -> str:
    val = os.getenv(env_var)
    if val:
        return val
    from pyspark.dbutils import DBUtils  # type: ignore
    return DBUtils(spark).secrets.get(scope=scope, key=key or env_var.lower())

In [4]:
spark.sql(f'CREATE VOLUME IF NOT EXISTS {UC_CATALOG}.{BRONZE_SCHEMA}.{VOLUME_NAME}')

def upload(path: str, payload, *, overwrite: bool = True) -> int:
    if isinstance(payload, (dict, list)):
        body = json.dumps(payload).encode('utf-8')
    elif isinstance(payload, str):
        body = payload.encode('utf-8')
    else:
        body = payload
    w.files.upload(path, body, overwrite=overwrite)
    return len(body)

def already_uploaded(path: str) -> bool:
    try:
        w.files.get_metadata(path)
        return True
    except Exception:
        return False

print('Volume helpers ready.')

Volume helpers ready.


## Step 1 — Read the seed list

Pulls artist names from `ARTIST_SEED_CSV`. The optional `rostr_handle`
column overrides the slugification (helpful for unusual names like P!nk).

In [5]:
from rostr_client import slugify

seed_path = ARTIST_SEED_CSV
if not os.path.isabs(seed_path):
    seed_path = os.path.join(os.getcwd(), ARTIST_SEED_CSV)

with open(seed_path, newline='') as f:
    reader = csv.DictReader(f)
    artists = []
    for row in reader:
        name = (row.get('artist_name') or '').strip()
        if not name:
            continue
        handle = (row.get('rostr_handle') or '').strip() or slugify(name)
        artists.append((name, handle))

print(f'Seed list: {len(artists)} artists')
for n, h in artists[:5]:
    print(f'  {n:<25s} -> {h}')
if len(artists) > 5:
    print(f'  ... +{len(artists)-5} more')

Seed list: 30 artists
  Olivia Rodrigo            -> oliviarodrigo
  Taylor Swift              -> taylorswift
  Justin Bieber             -> justinbieber
  Bad Bunny                 -> badbunny
  Beyonce                   -> beyonce
  ... +25 more


## Step 2 — Authenticate

In [6]:
from rostr_client import RostrClient, RostrAPIError

rostr = RostrClient(
    username=get_secret('ROSTR_USERNAME'),
    password=get_secret('ROSTR_PASSWORD'),
    cookie_path=os.getenv('ROSTR_COOKIE_PATH') or None,
    rate_limit_seconds=INGEST_RATE_LIMIT,
)
rostr.authenticate()
print('rostr.cc auth OK.')

rostr.cc auth OK.


## Step 3 — Per-artist fan-out

For each artist we land:
* `artists/{handle}.json` — the `/v1/artist/{handle}` payload (object as-is)
* `team/{handle}/{role}.json` — wrapped object `{handle, role, entries:[...]}` so bronze grain is one row per (artist, role)

Files already in the Volume are skipped (set `FORCE_REFRESH=true` to override).

In [7]:
def ingest_artist(name: str, handle: str) -> dict:
    summary = {'name': name, 'handle': handle, 'wrote': [], 'skipped': [], 'failed': []}

    artist_path = f'{ARTISTS_DIR}/{handle}.json'
    if not FORCE_REFRESH and already_uploaded(artist_path):
        summary['skipped'].append('artist')
    else:
        try:
            payload = rostr.get_artist(handle)
            upload(artist_path, payload)
            summary['wrote'].append('artist')
        except RostrAPIError as e:
            summary['failed'].append(('artist', str(e)[:120]))
            return summary
        except Exception as e:
            summary['failed'].append(('artist', str(e)[:120]))
            return summary

    for role in ROSTR_ROLES:
        team_path = f'{TEAM_DIR}/{handle}/{role}.json'
        if not FORCE_REFRESH and already_uploaded(team_path):
            summary['skipped'].append(f'team:{role}')
            continue
        try:
            entries = rostr.get_team(handle, role)
            upload(team_path, {'handle': handle, 'role': role, 'entries': entries})
            summary['wrote'].append(f'team:{role}')
        except Exception as e:
            summary['failed'].append((f'team:{role}', str(e)[:120]))
    return summary


results: list[dict] = []
with ThreadPoolExecutor(max_workers=INGEST_WORKERS) as exe:
    futures = {exe.submit(ingest_artist, name, handle): (name, handle) for name, handle in artists}
    done = 0
    for fut in as_completed(futures):
        name, handle = futures[fut]
        try:
            s = fut.result()
            results.append(s)
            tag = 'OK ' if not s['failed'] else 'ERR'
            print(f"  [{tag}] {name:<25s} wrote={len(s['wrote'])} skipped={len(s['skipped'])} failed={len(s['failed'])}")
            for kind, msg in s['failed']:
                print(f"          {kind}: {msg}")
        except Exception as e:
            print(f'  [HARD FAIL] {name}: {str(e)[:160]}')
        done += 1

print(f'\nIngest complete: {done} artists.')

  [OK ] Olivia Rodrigo            wrote=0 skipped=5 failed=0
  [OK ] Justin Bieber             wrote=0 skipped=5 failed=0
  [OK ] Bad Bunny                 wrote=0 skipped=5 failed=0
  [OK ] Taylor Swift              wrote=0 skipped=5 failed=0


  [OK ] The Weeknd                wrote=0 skipped=5 failed=0
  [OK ] Drake                     wrote=0 skipped=5 failed=0


  [OK ] Beyonce                   wrote=0 skipped=5 failed=0
  [OK ] Billie Eilish             wrote=0 skipped=5 failed=0
  [ERR] Ed Sheeran                wrote=0 skipped=0 failed=1
          artist: 404 on /v1/artist/edsheeran


  [OK ] Post Malone               wrote=0 skipped=5 failed=0
  [OK ] Ariana Grande             wrote=0 skipped=5 failed=0
  [OK ] Chappell Roan             wrote=0 skipped=5 failed=0


  [OK ] Sabrina Carpenter         wrote=0 skipped=5 failed=0


...

  [OK ] Harry Styles              wrote=0 skipped=5 failed=0
  [OK ] Doja Cat                  wrote=0 skipped=5 failed=0


  [OK ] Karol G                   wrote=0 skipped=5 failed=0
  [OK ] Morgan Wallen             wrote=0 skipped=5 failed=0


  [ERR] Megan Thee Stallion       wrote=0 skipped=0 failed=1
          artist: 404 on /v1/artist/megantheestallion
  [OK ] Tyler The Creator         wrote=0 skipped=5 failed=0
  [OK ] Zach Bryan                wrote=0 skipped=5 failed=0


  [OK ] Charli XCX                wrote=0 skipped=5 failed=0


  [OK ] Rihanna                   wrote=0 skipped=5 failed=0
  [OK ] Kanye West                wrote=0 skipped=5 failed=0
  [OK ] Lady Gaga                 wrote=0 skipped=5 failed=0

Ingest complete: 30 artists.


## Summary

In [8]:
def list_volume(p: str) -> int:
    try:
        return sum(1 for _ in w.files.list_directory_contents(p))
    except Exception:
        return 0

n_artists = list_volume(ARTISTS_DIR)
team_subdirs = list(w.files.list_directory_contents(TEAM_DIR)) if list_volume(TEAM_DIR) else []
n_team_files = sum(list_volume(d.path) for d in team_subdirs)

print('=' * 60)
print('INGEST SUMMARY')
print('=' * 60)
print(f'  artists/      : {n_artists} files')
print(f'  team/{{*}}/    : {n_team_files} files across {len(team_subdirs)} subdirs')
print(f'  Volume root   : {VOLUME_PATH}')
print('\nNext: run 02_bronze_autoloader.')

INGEST SUMMARY
  artists/      : 28 files
  team/{*}/    : 112 files across 28 subdirs
  Volume root   : /Volumes/alexander_booth/rostr_music_industry_bronze/raw_data

Next: run 02_bronze_autoloader.
